# 08 · 개발에서 고른 설정으로 A/B/C/D 다시 비교하기

07의 720 목표 대표 설명(실제 평균 약 566토큰)을 그대로 재사용한다. `g=20`, `K=5`, 최종 근거 최대 5개를 고정하고, 과거 평가와 개발에 사용하지 않은 문항에서 (초기 설정: **NQ 20＋HotpotQA 20문항**) 실제 검색과 GPT-6 Luna 답변 생성을 실행한다.

개발 92.5%는 **근거 확보율**이다. 여기서는 답변 정확도·전체 시간·생성기 입력·온라인 API 비용을 별도로 확인한다. D는 C에서 고른 정책을 Laya에 그대로 적용하는 비교군이며, D의 최적화 결과는 아니다.

## 1. 설정과 실행

문항 수는 `config/frozen_comparison.json`의 `questions_per_task`에서 변경한다. 두 데이터셋에 각각 적용한다. 호출 예산도 함께 확인한다.

기본 `RUN_FROZEN_COMPARISON=False`는 마지막 완료 결과를 검증하고 그림만 다시 만든다. 새 유료 평가를 원하면 아래 실행 셀에서 `True`로 바꾼다. `.env`의 OpenAI·TypeSafe 키를 사용하며 키 값은 출력하지 않는다. 05·06·07과 기존 라우팅·생성 엔진은 유지한다.

In [ ]:
from pathlib import Path
import sys, os, math, json, warnings
warnings.filterwarnings("ignore", message="IProgress not found.*", module="tqdm.auto")
import pandas as pd
import pyarrow  # Windows native DLL import order: before Torch.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, ticker
from IPython.display import display, Markdown

ROOT = Path.cwd() if (Path.cwd()/'rag_baselines').exists() else Path.cwd().parent
if not (ROOT/'rag_baselines').is_dir():
    raise FileNotFoundError('프로젝트 루트 또는 notebook 폴더에서 실행하세요.')
sys.path.insert(0, str(ROOT))
from rag_baselines.data import read_json, read_jsonl, write_json, file_sha256
from rag_baselines.evaluation import notebook_source_sha256
from rag_baselines.frozen_comparison import run_frozen_comparison, load_frozen_report, frozen_reference, METHODS

font_path = Path('C:/Windows/Fonts/malgun.ttf')
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=str(font_path)).get_name()
plt.rcParams.update({'font.size': 11, 'axes.unicode_minus': False,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'text.color': '#263746', 'axes.labelcolor': '#263746'})
COLORS = ['#416D95', '#BE7D32', '#537D59', '#8565A8']
NAMES = ['A · 기본 RAG', 'B · 재정렬 RAG', 'C · Jev 2단계', 'D · Laya 2단계']
SETTINGS = read_json(ROOT/'config/frozen_comparison.json')
planned, _, _, _, _ = frozen_reference(ROOT, SETTINGS)
display(pd.DataFrame([{
    '문항/데이터셋': SETTINGS['questions_per_task'], '전체 문항': 2*SETTINGS['questions_per_task'],
    '반복': SETTINGS['repeats'], '그룹 크기 g': planned['group_size'],
    '선택 그룹 K': planned['routing_policy']['selection_k'],
    '최대 최종 근거': planned['routing_policy']['evidence_k'],
    '설명 목표 토큰': planned['descriptor_target_tokens'],
    '설명 실제 평균 토큰': planned['descriptor_mean_tokens_proxy']
}]).round(2))

## 2. 동결 조건과 평가식

| 방법 | 실제 검색 경로 | 최종 생성기 |
|---|---|---|
| A | RetroMAE 질문 임베딩 → 공통 벡터 색인 검색 50개 → 상위 5개 원문 | GPT-6 Luna |
| B | A와 동일한 dense 후보 → DeBERTa 재정렬 → 상위 5개 원문 | 동일 |
| C | 동일한 100개 그룹의 대표 설명을 Jev로 판단 → 상위 5개 그룹의 원문 채점 → 최대 5개 원문 | 동일 |
| D | C와 동일한 설명·그룹·규칙, 판단기만 로컬 Laya | 동일 |

$E_q=\mathbf 1[\exists G\in\mathcal G_q:G\subseteq S_q]$는 필요한 근거 annotation 하나를 완전히 확보했는지,
$Y_q$는 정규화 Exact Match 성공 여부다. $R_{evidence}=\frac1Q\sum_q E_q$, $EM=\frac1Q\sum_qY_q$를 구분한다.

$$C_{API,q}=C_{judgment,q}+C_{generation,q},\qquad
T_q=T_{retrieval,q}+T_{generation,q}+T_{overhead,q}.$$

$U_{LLM,q}$는 **최종 GPT의 실제 API 청구 입력 토큰**이다. 판단기 입력 토큰은 다른 열에 저장하며 서로 합산해 연산량으로 해석하지 않는다. API 비용은 고정된 가격표로 usage를 환산한 추정 청구액이다. 문서 준비·로컬 GPU·사람 작업 비용은 별도로 표시한다.

이 정책은 세 길이 조건 중 개발 목표를 만족한 관측 설정이다. 전역 최적점·무손실 탐색·상수 지연을 뜻하지 않는다. 그룹 요약은 탐색에만 쓰고 GPT에는 **선택된 원문**을 전달한다.

In [ ]:
RUN_FROZEN_COMPARISON = False  # True면 새 유료 실행. 기본은 마지막 완료 결과 재분석.
RUN_FROZEN_COMPARISON = RUN_FROZEN_COMPARISON or os.environ.get('DECISION_LLM_RUN_FROZEN') == '1'
SYSTEM_RUN = None  # 지정하려면 results/frozen_* 경로. 기본: 마지막 완료 결과.
if RUN_FROZEN_COMPARISON:
    from collections import deque
    from IPython.display import clear_output
    messages = deque(maxlen=10)
    def progress(message):
        messages.append(message)
        clear_output(wait=True)
        print('08 실제 평가 진행 중 · 원시 로그는 호출마다 저장됩니다.')
        print('\n'.join(messages))
    RUN_DIR = run_frozen_comparison(ROOT, SETTINGS, progress=progress)
elif SYSTEM_RUN:
    RUN_DIR = (ROOT/SYSTEM_RUN).resolve()
else:
    pointer = ROOT/'results/current_frozen_run.txt'
    RUN_DIR = (ROOT/pointer.read_text(encoding='utf-8').strip()).resolve() if pointer.exists() else None

if RUN_DIR is None:
    print('아직 완료 결과가 없습니다. 위 RUN_FROZEN_COMPARISON=True로 새 평가를 실행하세요.')
else:
    df, summary, frozen, manifest = load_frozen_report(ROOT, RUN_DIR)
    records = read_jsonl(RUN_DIR/'system.jsonl')
    table = summary.set_index('method').loc[list(METHODS)]
    ANALYSIS = RUN_DIR/'analysis'
    ANALYSIS.mkdir(exist_ok=True)
    print('검증한 결과:', RUN_DIR)
    print('새 평가 문항:', int(table.questions.iloc[0]), '· 과거 평가 제외:', len(manifest['excluded_historical_eval_ids']))
    print('분석은 결과에 저장된 동결 설정을 사용합니다. 설정 파일 변경은 다음 실행에만 적용됩니다.')

## 3. 실제 비교 결과

모든 방법은 같은 새 문항과 같은 생성기 설정을 사용한다. 표의 정확도는 실제 생성 답변에 대한 EM이고, 근거 확보율은 선택한 원문이 정답 근거를 포함했는지 평가한 값이다.

In [ ]:
if RUN_DIR is not None:
    view = table[['questions', 'answer_em', 'answer_f1', 'complete_evidence_hit',
        'online_seconds', 'llm_input_tokens', 'evidence_count', 'online_api_cost_usd']].copy()
    view.index = NAMES
    view = view.rename(columns={'questions':'문항', 'answer_em':'답변 EM', 'answer_f1':'답변 F1',
        'complete_evidence_hit':'근거 확보율', 'online_seconds':'전체 시간(초)',
        'llm_input_tokens':'GPT 입력 토큰', 'evidence_count':'전달 근거 수', 'online_api_cost_usd':'API 비용(USD)'})
    display(view.style.format({'문항':'{:.0f}', '답변 EM':'{:.1%}', '답변 F1':'{:.1%}',
        '근거 확보율':'{:.1%}', '전체 시간(초)':'{:.3f}', 'GPT 입력 토큰':'{:.1f}',
        '전달 근거 수':'{:.2f}', 'API 비용(USD)':'${:.6f}'}))

## 4. 정확도·전체 시간·생성기 입력·온라인 비용

EM의 검은 선은 반복 1회일 때 95% Wilson 구간이다. 여러 번 반복하면 질문을 단위로 재표본 추출한 95% bootstrap 구간을 사용한다. 각 방법의 구간만으로 방법 간 우열을 확정하지 않고, 아래의 같은 문항 비교도 확인한다.

In [ ]:
if RUN_DIR is not None:
    positions = np.arange(4)
    fig, axes = plt.subplots(2, 2, figsize=(13.6, 9), layout='constrained')
    estimates, low, high = [], [], []
    rng = np.random.default_rng(20261004)
    for method in METHODS:
        values = df[df.method == method].groupby('question_id').answer_em.mean().to_numpy()
        p, count, z = values.mean(), len(values), 1.96
        if frozen['settings']['repeats'] == 1:
            center = (p+z*z/(2*count))/(1+z*z/count)
            half = z*math.sqrt(p*(1-p)/count+z*z/(4*count*count))/(1+z*z/count)
            lo, hi = center-half, center+half
        else:
            boot = rng.choice(values, size=(5000,count)).mean(axis=1)
            lo, hi = np.quantile(boot, [.025,.975])
        estimates.append(p); low.append(max(0,p-lo)); high.append(max(0,hi-p))
    bars = axes[0,0].bar(positions, estimates, color=COLORS, yerr=[low, high], capsize=4)
    axes[0,0].bar_label(bars, labels=[f'{v:.1%}' for v in estimates], padding=5)
    axes[0,0].set_ylim(0,1.15)
    axes[0,0].yaxis.set_major_formatter(ticker.PercentFormatter(1))
    axes[0,0].set_title('실제 생성 답변 정확도')
    axes[0,0].set_ylabel('EM · 95% 구간')
    for ax, column, title, unit, fmt in [
        (axes[0,1], 'online_seconds', '실제 전체 응답 시간', '초 / 질문', lambda v:f'{v:.2f}초'),
        (axes[1,0], 'llm_input_tokens', '최종 GPT 청구 입력', '입력 토큰 / 질문', lambda v:f'{v:,.0f}'),
        (axes[1,1], 'online_api_cost_usd', '온라인 API 비용 · 준비·GPU 제외', 'USD / 질문', lambda v:f'${v:.6f}')]:
        values = table[column].to_numpy()
        bars = ax.bar(positions, values, color=COLORS)
        ax.bar_label(bars, labels=[fmt(v) for v in values], padding=4, fontsize=10)
        ax.set_ylim(0,max(values.max()*1.25,1e-12)); ax.set_title(title); ax.set_ylabel(unit)
    for ax in axes.flat:
        ax.set_xticks(positions, NAMES, fontsize=10)
        ax.grid(axis='y', color='#E3E8EC', linewidth=.7); ax.set_axisbelow(True)
    fig.suptitle(f"08 · 새 평가 {int(table.questions.iloc[0])}문항 · {frozen['corpus_size']:,}문단 · "
        f"s 목표 {frozen['descriptor_target_tokens']} / 실제 평균 {frozen['descriptor_mean_tokens_proxy']:.0f} · "
        f"g={frozen['group_size']}, K={frozen['routing_policy']['selection_k']}", fontsize=13)
    fig.savefig(ANALYSIS/'frozen_comparison.png', dpi=160, bbox_inches='tight')
    plt.show()

## 5. 근거를 찾았는가, 찾은 근거로 답했는가

대표 설명 길이를 늘린 효과는 먼저 근거 확보율에서 확인한다. 정답 그룹을 선택하는 것, 그 안에서 필요한 원문을 전달하는 것, GPT가 정답을 생성하는 것은 별개다. NQ와 여러 자료가 필요한 HotpotQA를 나눠서 본다.

In [ ]:
if RUN_DIR is not None:
    segments = df.groupby(['task','method'], as_index=False).agg(
        questions=('question_id','nunique'), answer_em=('answer_em','mean'),
        complete_evidence_hit=('complete_evidence_hit','mean'), online_seconds=('online_seconds','mean'))
    segments.to_csv(ANALYSIS/'task_results.csv', index=False)
    fig, axes = plt.subplots(1,2,figsize=(13.2,4.8),layout='constrained')
    bars = axes[0].bar(np.arange(4), table.complete_evidence_hit, color=COLORS)
    axes[0].bar_label(bars, labels=[f'{v:.1%}' for v in table.complete_evidence_hit],padding=4)
    axes[0].set_xticks(np.arange(4),NAMES,fontsize=10)
    axes[0].set_title('최종 원문에 필요한 근거를 모두 포함한 비율')
    for index,task in enumerate(['nq','hotpotqa']):
        values = segments[segments.task == task].set_index('method').loc[list(METHODS)].answer_em
        bars = axes[1].bar(np.arange(4)+(index-.5)*.34,values,width=.34,color=COLORS,
            hatch='' if index==0 else '///',edgecolor='white',label='NQ' if index==0 else 'HotpotQA')
        axes[1].bar_label(bars,labels=[f'{v:.1%}' for v in values],padding=3,fontsize=9)
    axes[1].set_xticks(np.arange(4),NAMES,fontsize=10); axes[1].set_title('자료 유형별 실제 답변 EM'); axes[1].legend()
    for ax in axes:
        ax.set_ylim(0,1.12); ax.yaxis.set_major_formatter(ticker.PercentFormatter(1))
        ax.grid(axis='y',color='#E3E8EC'); ax.set_axisbelow(True)
    fig.savefig(ANALYSIS/'evidence_and_tasks.png',dpi=160,bbox_inches='tight'); plt.show()
    route = df[df.method.isin(['C-parallel','D-parallel'])].groupby('method').agg(
        macro_complete=('macro_complete','mean'), final_complete=('complete_evidence_hit','mean'),
        physical_http_requests=('physical_http_requests','mean'))
    display(route.round(4))

## 6. 토큰 감소와 비용 감소를 구분하기

최종 근거 수를 최대 5개로 맞췄으므로 C/D의 GPT 입력이 예전 트리 방식처럼 절반으로 줄어든다고 전제하지 않는다. 긴 대표 설명은 GPT가 아니라 판단기가 읽는다. 아래에서는 판단 입력·GPT 입력과 비용을 분리한다. D의 API 비용이 낮아도 로컬 GPU 비용을 0으로 측정한 것은 아니다.

In [ ]:
if RUN_DIR is not None:
    details = df.groupby('method').agg(
        judgment_input_tokens=('decision_input_tokens','mean'), gpt_input_tokens=('llm_input_tokens','mean'),
        original_evidence_tokens=('evidence_text_tokens','mean'), passages=('evidence_count','mean'),
        judgment_api_usd=('retrieval_api_cost_usd','mean'), generation_api_usd=('generation_api_cost_usd','mean'),
        requests=('physical_http_requests','mean'), local_peak_bytes=('local_peak_allocated_bytes','max')).loc[list(METHODS)]
    details.to_csv(ANALYSIS/'input_and_cost_components.csv')
    token_display = details[['judgment_input_tokens','gpt_input_tokens','original_evidence_tokens','passages','requests']].rename(columns={
        'judgment_input_tokens':'판단모델 입력 토큰', 'gpt_input_tokens':'GPT 입력 토큰',
        'original_evidence_tokens':'근거 원문 토큰', 'passages':'원문 개수', 'requests':'판단 API 요청 수'})
    cost_display = details[['judgment_api_usd','generation_api_usd']].rename(columns={
        'judgment_api_usd':'판단 API (USD)', 'generation_api_usd':'생성 API (USD)'})
    cost_display['최대 로컬 할당 (GiB)'] = details.local_peak_bytes / 2**30
    display(token_display.style.format('{:.1f}', na_rep='해당 없음'))
    display(cost_display.style.format({
        '판단 API (USD)':'{:.6f}', '생성 API (USD)':'{:.6f}', '최대 로컬 할당 (GiB)':'{:.2f}'}, na_rep='해당 없음'))
    fig, ax = plt.subplots(figsize=(10.4,4.8),layout='constrained')
    positions = np.arange(4)
    ax.bar(positions,details.generation_api_usd,color=COLORS,label='GPT 생성')
    ax.bar(positions,details.judgment_api_usd,bottom=details.generation_api_usd,color=COLORS,
        hatch='///',edgecolor='white',label='판단 API')
    ax.set_xticks(positions,NAMES); ax.set_ylabel('USD / 질문'); ax.set_ylim(bottom=0)
    ax.set_title('온라인 청구 비용 구성 · 로컬 GPU 및 준비 비용 별도'); ax.legend()
    ax.grid(axis='y',color='#E3E8EC'); ax.set_axisbelow(True)
    fig.savefig(ANALYSIS/'api_cost_components.png',dpi=160,bbox_inches='tight'); plt.show()
    preparation = read_json(RUN_DIR/'preparation.json')
    ledger = read_jsonl(ROOT/manifest['raw_log']/'api_calls.jsonl')
    warmup_fee = sum(r['api_cost_usd'] for r in ledger if r['phase']=='warmup')
    display(Markdown(f"**이 실행의 새 대표 설명 생성: {preparation['actual_new_description_calls']}회.** "
        f"재사용한 설명의 과거 준비 API 비용은 USD {preparation['descriptions']['api_cost_usd_total']:.6f}; "
        f"기존 분류 준비는 별도 원시 영수증에 보존된다. 이번 warm-up API 비용 USD {warmup_fee:.6f}는 온라인 평균에서 제외한다."))

## 7. 같은 문항에서의 차이

C가 맞히고 비교군이 틀린 문항과 그 반대 문항을 함께 계산한다. EM 차이의 95% 구간은 질문 단위의 쌍을 유지한 bootstrap이다. 새 08과 과거 07은 평가 문항이 다르므로 두 그래프의 차이를 요약 길이의 인과 효과로 계산하지 않는다.

In [ ]:
if RUN_DIR is not None:
    paired = df.groupby(['question_id','method']).answer_em.mean().unstack('method')
    changes=[]
    rng = np.random.default_rng(20261004)
    for candidate,baseline in [('C-parallel','A'),('C-parallel','B'),('D-parallel','B')]:
        delta = (paired[candidate]-paired[baseline]).to_numpy()
        boot = rng.choice(delta,size=(5000,len(delta))).mean(axis=1)
        low,high = np.quantile(boot,[.025,.975])
        changes.append({'candidate':candidate,'baseline':baseline,'questions':len(delta),
            'candidate_better':int((delta>0).sum()),'candidate_worse':int((delta<0).sum()),
            'equal':int((delta==0).sum()),'em_difference':delta.mean(),'bootstrap_low':low,'bootstrap_high':high})
    paired_changes = pd.DataFrame(changes)
    paired_changes.to_csv(ANALYSIS/'paired_answer_changes.csv',index=False)
    display(paired_changes.round(4))

## 8. 관측 결과의 해석과 원시 근거

결론은 아래 실제 측정값에서 자동 계산한다. 2,000문단의 정답 포함 축소 코퍼스와 소규모 새 평가의 결과이며, 전체 Wikipedia·수만 문서·다른 GPU에서의 성능을 보장하지 않는다. 평가 결과를 보고 설정을 다시 고르면 이 문항들은 이후 개발에 사용한 것으로 취급하고 다음 평가를 분리해야 한다.

In [ ]:
if RUN_DIR is not None:
    a,b,c,d = (table.loc[m] for m in METHODS)
    display(Markdown(
        f"**새 {int(c.questions)}문항:** A/B/C/D 답변 EM은 {a.answer_em:.1%}/{b.answer_em:.1%}/{c.answer_em:.1%}/{d.answer_em:.1%}. "
        f"C의 근거 확보율은 {c.complete_evidence_hit:.1%}, B는 {b.complete_evidence_hit:.1%}. "
        f"C의 전체 시간은 {c.online_seconds:.2f}초, 온라인 API 비용은 USD {c.online_api_cost_usd:.6f}/질문이다. "
        f"B 대비 시간 {c.online_seconds/b.online_seconds:.2f}배, API 비용 {c.online_api_cost_usd/b.online_api_cost_usd:.1f}배다. "
        f"최종 GPT 입력은 C {c.llm_input_tokens:.1f}, A {a.llm_input_tokens:.1f}토큰으로 {100*(1-c.llm_input_tokens/a.llm_input_tokens):.1f}% 감소했다."
    ))
    checks = {'completed_report_and_physical_ledger_verified': True,
        'evaluation_questions':int(c.questions), 'all_method_records':len(df),
        'prior_eval_excluded':len(manifest['excluded_historical_eval_ids']),
        'dev_eval_overlap':bool(set(manifest['question_ids']) & set(manifest['development_question_ids'])),
        'reported_decision_truncations':int(df.input_truncated.sum()),
        'new_description_generation_calls':preparation['actual_new_description_calls'],
        'actual_frozen_descriptor_mean_tokens_proxy':frozen['descriptor_mean_tokens_proxy'],
        'scope':'Actual answer generation on fresh held-out questions; no superiority/constant-time guarantee.'}
    write_json(ANALYSIS/'validation_checks.json',checks)
    write_json(ANALYSIS/'analysis_manifest.json',{
        'run':str(RUN_DIR),'system_sha256':file_sha256(RUN_DIR/'system.jsonl'),
        'frozen_policy_sha256':file_sha256(RUN_DIR/'frozen_policy.json'),
        'notebook_source_sha256':notebook_source_sha256(ROOT/'notebook/08_compare_frozen_routing.ipynb'),
        'exports_sha256':{p.name:file_sha256(p) for p in ANALYSIS.iterdir() if p.is_file() and p.name!='analysis_manifest.json'}})
    print('질문별 결과:',RUN_DIR/'questions.csv')
    print('질문·원문·생성 답변:',RUN_DIR/'system.jsonl')
    print('API 요청·응답·로컬 추론:',ROOT/manifest['raw_log'])
    print('비교 그림과 검증:',ANALYSIS)